In [ ]:
%cd ~/cdv
import os

# os.environ['CUDA_VISIBLE_DEVICES'] = ''
import numpy as np
import jax.numpy as jnp
import jax
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

import rho_plus as rp

is_dark = False
theme, cs = rp.mpl_setup(is_dark)
rp.plotly_setup(is_dark)

In [ ]:
from ase import Atoms

from agox import AGOX
from agox.acquisitors import LowerConfidenceBoundAcquisitor
from agox.collectors import ParallelCollector
from agox.databases import Database
from agox.environments import Environment
from agox.evaluators import LocalOptimizationEvaluator
from agox.generators import RandomGenerator, RattleGenerator
from agox.models.descriptors.fingerprint import Fingerprint
from agox.models.GPR import GPR
from agox.models.GPR.kernels import RBF, Noise
from agox.models.GPR.kernels import Constant as C
from agox.models.GPR.priors import Repulsive
from agox.postprocessors import ParallelRelaxPostprocess
from agox.samplers import KMeansSampler

from facet.io.structure_calc import FacetModel

# Manually set seed and database-index
seed = 41
database_index = 0

##############################################################################
# Calculator
##############################################################################

from facet.io.ase_calc import FacetCalculator

calc = FacetCalculator(FacetModel.new_facet())

##############################################################################
# System & general settings:
##############################################################################

template = Atoms("", cell=np.eye(3) * 12)
confinement_cell = np.eye(3) * 6
confinement_corner = np.array([3, 3, 3])
environment = Environment(
    template=template,
    symbols="Au8Ni8",
    confinement_cell=confinement_cell,
    confinement_corner=confinement_corner,
)

# Database
db_path = "db{}.db".format(database_index)  # From input argument!
database = Database(filename=db_path, order=5)

##############################################################################
# Search Settings:
##############################################################################

# Setup a ML model.
descriptor = Fingerprint(environment=environment)
beta = 0.01
k0 = C(beta, (beta, beta)) * RBF()
k1 = C(1 - beta, (1 - beta, 1 - beta)) * RBF()
kernel = C(5000, (1, 1e5)) * (k0 + k1) + Noise(0.01, (0.01, 0.01))
model = GPR(descriptor=descriptor, kernel=kernel, database=database, prior=Repulsive())

# Sampler to choose candidates to modify using the generators.
sample_size = 10
sampler = KMeansSampler(descriptor=descriptor, database=database, sample_size=sample_size)

# Generators to produce candidates structures
rattle_generator = RattleGenerator(**environment.get_confinement())
random_generator = RandomGenerator(**environment.get_confinement())

# Dict specificies how many candidates are created with and the dict-keys are iterations.
generators = [random_generator, rattle_generator]
num_candidates = {0: [10, 0], 5: [3, 7]}

# Collector creates a number of structures in each iteration.
collector = ParallelCollector(
    generators=generators,
    sampler=sampler,
    environment=environment,
    num_candidates=num_candidates,
    order=1,
)

# Acquisitor to choose a candidate to evaluate in the real potential.
acquisitor = LowerConfidenceBoundAcquisitor(model=model, kappa=2, order=3)

# Number of steps is very low - should be set higher for a real search!
relaxer = ParallelRelaxPostprocess(
    model=acquisitor.get_acquisition_calculator(),
    constraints=environment.get_constraints(),
    optimizer_run_kwargs={"steps": 5},
    start_relax=8,
    order=2,
)

import traceback
from uuid import uuid4

import numpy as np
from ase.calculators.singlepoint import SinglePointCalculator
from ase.constraints import FixAtoms
from ase.optimize.bfgs import BFGS

from agox.evaluators.ABC_evaluator import EvaluatorBaseClass


class SinglePotentialEvalulator(EvaluatorBaseClass):
    """
    Evaluator that performs a local optimization of a candidate using a ASE BFGS optimizer.

    Parameters:
    -----------
    calculator: ASE calculator
        The calculator to use for the evaluation.
    optimizer: ASE optimizer, optional
        The optimizer to use for the local optimization. Default is BFGS.
    optimizer_run_kwargs: dict, optional
        The keyword arguments to pass to the optimizer.run() method.
    optimizer_kwargs: dict, optional
        The keyword arguments to pass to the optimizer constructor.
    fix_template: bool, optional
        Whether to fix the template atoms during the optimization. Default is True.
    constraints: list, optional
        List of constraints to apply during the optimization.
    store_trajectory: bool, optional
        Whether to store the trajectory of the optimization. Default is True.
        If False only the last step is stored.
    """
    name = "SinglePotentialCalculator"

    def __init__(
        self,
        calculator,                
        fix_template=True,
        constraints=[],
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.calculator = calculator
        # Constraints:
        self.constraints = constraints
        self.fix_template = fix_template

    def evaluate_candidate(self, candidate):
        candidate.calc = self.calculator

        try:            
            E = candidate.get_potential_energy(apply_constraint=False)                    
            self.evaluated_candidates.append(candidate)

        except Exception as e:
            self.writer("Energy calculation failed with exception: {}".format(e))
            traceback.print_exc()
            return False

        E = candidate.get_potential_energy(apply_constraint=False)        
        self.writer(f"Final energy of candidate = {E:5.3f}")
        calc = SinglePointCalculator(candidate, energy=E)
        candidate.calc = calc

        return True

    def apply_constraints(self, candidate):
        constraints = [] + self.constraints
        if self.fix_template:
            constraints.append(self.get_template_constraint(candidate))

        for constraint in constraints:
            if hasattr(constraint, "reset"):
                constraint.reset()

        candidate.set_constraint(constraints)

    def get_template_constraint(self, candidate):
        return FixAtoms(indices=np.arange(len(candidate.template)))



# Evaluator to evaluate the candidates in the real potential.
evaluator = SinglePotentialEvalulator(calculator=calc)

##############################################################################
# Let get the show running!
##############################################################################

# The oder of things here does not matter. But it can be simpler to understand
# what the expected behaviour is if they are put in order.
agox = AGOX(collector, relaxer, acquisitor, evaluator, database, seed=seed)

agox.run(N_iterations=10)

In [ ]:
evaluator